# 角点预测网络
单张 RGB 图片经过 ResNet-18 提取特征，再上采样得到八个角点的热图。按照 `Dataset → DataLoader → nn.Module → loss / optimizer → 训练循环 → evaluation` 的顺序组织。
当前使用已有 20 张图片检查数据读取和前向、反向计算；训练循环放在 Markdown 代码块中，搭建阶段先不执行。

In [1]:
from pathlib import Path
import sys

import torch
from torch import nn, optim
from torch.utils.data import DataLoader

project_dir = Path("S:/Deep Learning/coding_test_for_research")
sys.path.append(str(project_dir / "corner_prediction"))

from dataset import CornerDataset
from model import CornerPredictor

torch.manual_seed(42)
device = torch.device("cuda")
print(device)

cuda


## Dataset / DataLoader
`CornerDataset` 写在 [dataset.py](dataset.py)：按文件名配对 RGB 和 `.npz`，返回 `(x,y)`。RGB 转为 Tensor 后做 ImageNet normalization；热图保持原值。这里不裁剪、不缩放，图片和标签的像素坐标一致。

In [2]:
scene_dir = project_dir / "datasets/action4/train_pbr/000000"
dataset = CornerDataset(scene_dir)
train_loader = DataLoader(dataset, batch_size=2, shuffle=True, num_workers=0)

x, y = next(iter(train_loader))
print("图片数量:", len(dataset))
print("x:", x.shape)  # (N, 3, 480, 640)
print("y:", y.shape)  # (N, 8, 480, 640)

图片数量: 20
x: torch.Size([2, 3, 480, 640])
y: torch.Size([2, 8, 480, 640])


## nn.Module
网络定义在 [model.py](model.py)，沿用 `__init__` 定义层、`forward` 顺序计算的写法。`encoder` 保留 ResNet 的卷积和残差块，去掉用于分类的全局池化和全连接层。`decoder` 用五层转置卷积逐次恢复分辨率，`heatmap_head` 将 16 个特征通道映射为 8 个角点通道，最后通过 Sigmoid 得到 `[0,1]` 的热图。

| 部分 | 输出尺寸（省略 batch） |
| --- | --- |
| RGB 输入 | `3×480×640` |
| ResNet-18 encoder | `512×15×20` |
| decoder 第一次上采样 | `256×30×40` |
| 第二次上采样 | `128×60×80` |
| 第三次上采样 | `64×120×160` |
| 第四次上采样 | `32×240×320` |
| 第五次上采样 | `16×480×640` |
| heatmap head | `8×480×640` |

`weights=None` 为随机初始化，本次不下载预训练权重。当前输入的高、宽都是 32 的倍数，五次上采样后恢复原尺寸。

In [3]:
model = CornerPredictor(num_corners=8, weights=None).to(device)
num_params = sum(p.numel() for p in model.parameters())
print("参数量:", num_params)

参数量: 13971608


## loss / optimizer
先使用 BoxDreamer 的 coarse loss：对预测热图和真实热图计算 SmoothL1。这里先验证热图回归，角点坐标的 fine loss 留到训练部分接入。

In [4]:
loss_fn = nn.SmoothL1Loss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

## 前向与反向计算
取一个真实 batch，检查输出和标签形状，并执行 `backward()`。这一格只计算梯度，没有调用 `optimizer.step()`，因此不更新参数。

In [5]:
model.train()
x = x.to(device)
y = y.to(device)

pred = model(x)  # (N, 8, 480, 640)
loss = loss_fn(pred, y)
optimizer.zero_grad()
loss.backward()

print("pred:", pred.shape)
print("值域:", pred.min().item(), pred.max().item())
print("loss:", loss.item())
print("encoder 梯度范数:", model.encoder[0].weight.grad.norm().item())
print("head 梯度范数:", model.heatmap_head[0].weight.grad.norm().item())

pred: torch.Size([2, 8, 480, 640])
值域: 0.002041865838691592 0.9977194666862488
loss: 0.06038812920451164
encoder 梯度范数: 0.025284679606556892
head 梯度范数: 0.023267073556780815


## 训练循环
后续做 20 张图片的过拟合实验时，使用下面的循环：
```python
num_epochs = 100
for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.shape[0]

    print(epoch + 1, total_loss / len(dataset))
```

## Evaluation
调用 `eval()` 后，用 `no_grad()` 预测热图。每个通道取最大响应位置，得到 `(x,y)`。八个通道仍对应原始三维角点的 0–7 编号。当前网络尚未训练，这里只检查输出格式。

In [6]:
model.eval()
with torch.no_grad():
    pred = model(x)
    N, C, H, W = pred.shape
    indices = pred.flatten(start_dim=2).argmax(dim=2)  # (N, 8)
    corners_xy = torch.stack((indices % W, indices // W), dim=-1)  # (N, 8, 2)

print("corners_xy:", corners_xy.shape)

corners_xy: torch.Size([2, 8, 2])


## 保存模型
训练结束后保存 `state_dict`：
```python
torch.save(model.state_dict(), project_dir / "corner_prediction/corner_predictor.pt")
```
资料：[BoxDreamer §3.3](https://arxiv.org/html/2504.07955v2#S3.SS3)、[torchvision ResNet-18](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)。